# Explaining Nearest-Neighbor Classifiers

When explaining nearest-neighbor classifiers, the goal is, given a model trained on some training dataset $D$, to quantify how much each training data point $z_i \in D$ contributes to the model predicting the 'explanation class' $y_\text{explain}$ on some explanation point $x_\text{explain}$. For this purpose, in the case of a simple $k$-nearest neighbors classifier, we can define the utility of a coalition $S$ as the likelihood of predicting $y_\text{explain}$ [Jia19]_:
$$
    \nu(S) = \frac{1}{k} \sum_{k=1}^{\min\{k, |S|\}} \chi(y_{\alpha_k} = y_\text{explain}),
$$
where $y_{\alpha_k}$ is the index of the $k$-nearest training point to $x_\text{explain}$ in the coalition $S$, and $\chi(P)$ is the indicator function, defined as
$$
    \chi(P) = \left\lbrace
        \begin{array}{ll}
            1 & \text{if } P \\
            0 & \text{otherwise.} \\
        \end{array}
    \right.
$$

While `shapiq` already offers a way to explain nearest-neighbor models using `ExactComputer`, it requires performing an exhaustive search of all coalitions, resulting in an exponential runtime. However, special properties of nearest-neighbor models can be exploited for designing log-linear-time and quadratic-time algorithms, which are implemented in this library.

We implement explainers for three kinds of nearest-neighbor classifiers, based on recent pulications in the area of Explainable AI and Data Valuation:

- k-nearest neighbor classifiers (based on [Jia19]_),
- weighted k-nearest neighbor classifiers (based on [Wng24]_), and
- threshold nearest neighbor classifiers (based on [Wng23]_).

Let's start with some setup. First, we define a helper function to plot a training dataset, which we'll use later.

In [ ]:
import matplotlib.pyplot as plt


def plot_datasets(ax, X_train, y_train, title=None) -> None:
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    if title is not None:
        ax.set_title(title)
    ax.scatter(
        X_train[:, 0],
        X_train[:, 1],
        c=[colors[i] for i in y_train],
        label="Training Points",
        marker="o",
    )

    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=colors[i],
            markersize=10,
            label=f"Class {i}",
        )
        for i in set(y_train)
    ]
    ax.legend(handles=handles, loc="upper right", title="Data Points")

    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

To illustrate our Explainers, we generate a synthetic dataset with some non-linearity, making it suitable for nearest-neighbor classification. The dataset has two classes and just two features to simplify the presentation.

In [ ]:
from sklearn.datasets import make_classification

X_train, y_train = make_classification(
    n_samples=30,
    n_features=2,
    n_redundant=0,
    n_clusters_per_class=1,
    n_informative=2,
    n_classes=2,
    random_state=45,
)

fig, ax = plt.subplots()
plot_datasets(ax, X_train, y_train)
print(f"Size of training dataset: {X_train.shape[0]}")

Now, let's fit a KNN model to the training data. Then, we can define a explanation data point $x_\text{explain}$ and get its predicted class $\hat y_\text{explain}$.

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

x_explain = np.array([[-0.75, -0.4]])
y_explain_pred = model.predict(x_explain)
y_explain_pred

## Explaining a Single Data Point

Next, let's create an explainer for the model we just defined. The `KNNExplainer` class is the main interface to our explainers. It automatically selects the right explainer subclass depending on the type of model provided. In this case, we are passing an unweighted KNN model, so `NormalKNNExplainer` will be selected.

In [ ]:
from shapiq_student import KNNExplainer

explainer = KNNExplainer(model, class_index=y_explain_pred)
explainer.__class__

Note that we set `class_index=y_test_pred`, since for now, we want to quantify the contribution of the training data to the class that was actually predicted. (We could also set a different class index if we wished to see how much the data points contribute shifting the prediction towards another class.)

In order to obtain an explanation, we simply call the `explain()` method of the explainer with the test datapoint. This will return an `InteractionValues` object containing the Shapley Values for each datapoint in the training data set. We can convert this to a simple numpy array, since we only have interactions of order 1.

In [ ]:
from shapiq_student.explainer.knn import interaction_values_to_array

iv = explainer.explain(x_explain)
sv = interaction_values_to_array(iv)
print(sv.shape[0])
print(sv)

The `explain()` method implements the algorithm proposed in [Jia19]_\ for calculating Shapley Values for unweighted $k$-nearest neighbor models in **log-linear time**. To do so, it inspects the training data of the `sklearn` model that was passed in the constructor, sorts the training data points by their distance to $x_\text{test}$ and finally computes Shapley Values in linear time.

Just an array of floats is not very helpful, but we can visualize the Shapley Values by plotting the training data set again, this time setting the size of each dot according to the Shapley Value attributed to the training data point it represents. The `shapiq_student` library provides the function `plot_knn_shapley_2d` for just this purpose. Note that the dots' sizes will be set according to their _absolute_ Shapley Value. This doesn't make us lose any information, though, since the Shapley Value will be positive for exactly those training data points whose class label agrees with the class $y_i$ being explained, and negative for all other training data points. This is because adding such an 'incorrect' data point to a coalition can always only make the predicition worse, never improve it.

In [ ]:
from contextlib import suppress
from importlib import reload
import sys

with suppress(KeyError):
    reload(sys.modules["shapiq_student.plot.knn"])

from shapiq_student.plot.knn import plot_knn_shapley_2d

plot_knn_shapley_2d(X_train, y_train, sv, set(y_train), x_explain, scale=1)

The plotting function represents each training data point as a filled circle, colored according to its class and scaled by its absolute Shapley Value, together with a black ring whose size represents the maximum absolute Shapley Value. This allows putting each data point's Shapley Value into scale.

As we can observe, training data points that are closer to the explanation point have higher absolute Shapley Values, while those that are farther away have lower Shapley Values. This makes sense, since those points that are closest to the explanation point have the highest chance of being among the $k$-nearest points, thereby influencing the prediction of the model, while points farther away can rarely change the prediction.

Note, again, that all data points of the class with index `0` have positive Shapley Values and all others have negative Shapley Values, since we chose `y_test_pred==0` as our `class_index`. We can verify this fact by filtering the array of Shapley Values by the corresponding class:

In [ ]:
print(sv[y_train == y_explain_pred])
print(sv[y_train != y_explain_pred])